<a href="https://colab.research.google.com/github/Sothiphak/llm-prompting-tp2/blob/main/LLM_Prompting_Ollama_Colab_TP_completed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prompting Techniques with Ollama

Prompting steers a pretrained LLM at inference time, with **no weight updates**. In this notebook you will:

1. See four techniques in action: **few-shot**, **chain-of-thought**, **self-consistency**, **RAG**
2. Practice them yourself in short exercises

| Technique | Idea |
|---|---|
| Zero/Few-shot | Give instructions, optionally with examples |
| Chain-of-Thought (CoT) | Ask for step-by-step reasoning before the answer |
| Self-Consistency | Sample several CoT chains, majority-vote the answer |
| RAG | Retrieve relevant documents and put them in the prompt |

## Setup
Run once (Colab with GPU runtime). It installs Ollama and pulls a small model.

In [1]:
%pip install -q ollama scikit-learn numpy
!sudo apt-get install -y -qq zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
!nohup ollama serve > ollama.log 2>&1 &
import time; time.sleep(5)
!ollama pull gemma4:e2b

debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 6.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 



In [2]:
import ollama, re
from collections import Counter

MODEL = "gemma4:e2b"

def generate(prompt, max_tokens=150, temperature=0.0):
    """Send a prompt to the model and return its text. temperature=0 is deterministic."""
    r = ollama.generate(model=MODEL, prompt=prompt, think=False,
                        options={"temperature": temperature, "num_predict": max_tokens})
    return r["response"].strip()

print(generate("Say hello in one short sentence."))

Hello!


## Demo 1: Zero-shot vs. Few-shot
Few-shot adds input→output examples so the model copies the format.

In [3]:
review = "The pacing was dreadful, but the soundtrack was gorgeous."
task = "Classify the review as Positive, Negative, or Mixed. Reply with one word."

zero_shot = f"{task}\n\nReview: {review}\nLabel:"

few_shot = f"""{task}

Review: A masterpiece of modern cinema.
Label: Positive

Review: I fell asleep halfway through.
Label: Negative

Review: Great acting, but the plot made no sense.
Label: Mixed

Review: {review}
Label:"""

print("Zero-shot:", generate(zero_shot, max_tokens=10))
print("Few-shot :", generate(few_shot, max_tokens=10))

Zero-shot: Mixed
Few-shot : Mixed


## Demo 2: Chain-of-Thought
Asking the model to reason step by step helps on multi-step problems.

In [4]:
question = ("A store had 50 shirts. It sold 18 on Monday, got a shipment of 30, "
            "then sold half of what it had. How many shirts are left?")   # answer: 31

print("Direct:\n", generate(f"Q: {question}\nGive only the final number.\nA:", max_tokens=10))
print("\nZero-shot CoT:\n", generate(f"Q: {question}\nA: Let's think step by step.", max_tokens=200))

Direct:
 36

Zero-shot CoT:
 Here is the step-by-step solution:

**1. Starting Inventory:**
The store begins with **50** shirts.

**2. Sales on Monday:**
They sold 18 shirts.
*Shirts remaining after Monday's sales:* $50 - 18 = 32$ shirts.

**3. Shipment Received:**
They received a shipment of 30 shirts.
*Total shirts after the shipment:* $32 + 30 = 62$ shirts.

**4. Second Sale:**
They sold half of what they had (half of 62).
*Shirts sold in the second sale:* $62 / 2 = 31$ shirts.

**5. Final Count:**
*Shirts left:* $62 - 31 = 31$ shirts.

**Answer:** There are **31** shirts left.


## Demo 3: Self-Consistency
Sample several reasoning chains (temperature > 0) and take the most common final answer.

In [5]:
def final_number(text):
    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    return nums[-1] if nums else None

prompt = (f"Q: {question}\nThink step by step, then end with 'The answer is <number>.'\nA:")
answers = []
for _ in range(5):
    out = generate(prompt, max_tokens=250, temperature=0.8)
    answers.append(final_number(out))

print("Votes :", Counter(answers))
print("Winner:", Counter(answers).most_common(1)[0][0], "(truth: 31)")

Votes : Counter({'31': 5})
Winner: 31 (truth: 31)


## Demo 4: RAG
Retrieve the most relevant document (TF-IDF here; real systems use embeddings + a vector DB) and add it to the prompt.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

docs = [
    "Acme Corp's refund policy: items can be returned within 45 days with a receipt.",
    "Acme Corp support hours are 9am-5pm ICT, Monday to Friday.",
    "Acme Corp's Pro plan costs $29 per month and includes 5 seats.",
    "Acme Corp headquarters are located in Phnom Penh, Cambodia.",
]
vec = TfidfVectorizer().fit(docs)
doc_vecs = vec.transform(docs)

def retrieve(query, k=1):
    scores = cosine_similarity(vec.transform([query]), doc_vecs).ravel()
    return [docs[i] for i in scores.argsort()[::-1][:k]]

def rag_answer(query, k=1):
    context = "\n".join(retrieve(query, k))
    return generate(f"Answer using only the context.\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:")

q = "How many days do I have to return an item at Acme Corp?"
print("No context:", generate(f"Question: {q}\nAnswer:", max_tokens=60))
print("With RAG  :", rag_answer(q))

No context: Please provide me with more information. I do not have access to specific return policies for Acme Corp.

To find out how many days you have to return an item, you will need to check one of the following:

* **The Acme Corp website:** Look for their "Returns," "Shipping
With RAG  : The context does not state how many days you have to return an item at Acme Corp.


---
# Hands-on Practice
Complete the `TODO`s. Re-run each cell and compare outputs. Small models are noisy, so try a few variations.

### Exercise 1: Few-shot extraction
Make the model turn a sentence into JSON with keys `name`, `city`, `age`. First try zero-shot, then add **2 examples** to get a clean, consistent format.

In [7]:
text = "Sokha, a 29-year-old engineer, just moved to Siem Reap."

# Zero-shot baseline (no examples)
zero = f"""Extract name, city, and age as JSON.

Text: {text}
JSON:"""

# Few-shot: two examples fix the format (valid JSON, exact keys, age as a number)
few = f"""Extract name, city, and age as JSON. Reply with JSON only.

Text: Dara, a 34-year-old teacher, lives in Battambang.
JSON: {{"name": "Dara", "city": "Battambang", "age": 34}}

Text: Srey Neang, who is 22, recently started a job in Kampot.
JSON: {{"name": "Srey Neang", "city": "Kampot", "age": 22}}

Text: {text}
JSON:"""

print("Zero-shot:\n", generate(zero, max_tokens=60))
print("\nFew-shot:\n", generate(few, max_tokens=60))

Zero-shot:
 ```json
{
  "name": "Sokha",
  "city": "Siem Reap",
  "age": 29
}
```

Few-shot:
 {"name": "Sokha", "city": "Siem Reap", "age": 29}


### Exercise 2: Chain-of-Thought
The puzzle below often trips up direct answers. Compare a **direct** prompt with a **CoT** prompt. Which is right? (Answer: 0.05, i.e. 5 cents)

In [13]:
puzzle = "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much is the ball?"

direct = f"Q: {puzzle}\nGive only the final answer, no explanation.\nA:"
cot    = (f"Q: {puzzle}\n"
          "Let's think step by step. Define the unknown, write the equation, solve it, "
          "then finish with 'Answer: <amount in dollars>'.\nA:")

print("Direct:", generate(direct, max_tokens=40))
print("CoT   :", generate(cot, max_tokens=500))

Direct: $0.05
CoT   : Here is the step-by-step solution:

**1. Define the unknown:**
Let $B$ be the cost of the bat (in dollars).
Let $L$ be the cost of the ball (in dollars).

**2. Write the equations based on the problem statement:**
We are given two pieces of information:
* The total cost is $1.10:
   $$B + L = 1.10 \quad \text{(Equation 1)}$$
* The bat costs $1.00 more than the ball:
   $$B = L + 1.00 \quad \text{(Equation 2)}$$

**3. Solve the system of equations:**
We can use the substitution method. Substitute the expression for $B$ from Equation 2 into Equation 1:
$$(L + 1.00) + L = 1.10$$

Combine the $L$ terms:
$$2L + 1.00 = 1.10$$

Subtract $1.00$ from both sides:
$$2L = 1.10 - 1.00$$
$$2L = 0.10$$

Divide by 2 to find the cost of the ball ($L$):
$$L = \frac{0.10}{2}$$
$$L = 0.05$$

**4. Check the answer (Optional but recommended):**
If the ball ($L$) costs $\$0.05$, then the bat ($B$) costs $\$1.00$ more:
$B = 0.05 + 1.00 = 1.05$
Check the total cost: $B + L = 1.05 + 0.05 =

### Exercise 3: Self-Consistency
Write a function `self_consistent(prompt, k)` that samples `k` answers at `temperature=0.8` and returns the majority answer. Test it on the puzzle above with `k=1` vs `k=7`. Does voting help?

*Hint:* ask the model to end with `Answer: <number>` so extracting the answer is easy.

In [12]:
cot_prompt = (f"Q: {puzzle}\n"
              "Think step by step, but keep the reasoning under 80 words, "
              "then end with exactly 'Answer: <number in dollars>'.\nA:")

def extract_answer(text):
    """Take the number after the last 'Answer:'; None if the chain never reached an answer.
    Cents are converted to dollars so 0.05 and 5 cents count as the same vote."""
    m = re.findall(r"Answer:\s*\*{0,2}\s*\$?\s*(-?\d+(?:\.\d+)?)\s*(cents?|¢)?", text, flags=re.I)
    if not m:
        return None
    value, unit = m[-1]
    value = float(value)
    return round(value / 100 if unit else value, 2)

def self_consistent(prompt, k=5):
    """Sample k chains at temperature 0.8, extract each final answer, return the majority."""
    answers = [extract_answer(generate(prompt, max_tokens=500, temperature=0.8)) for _ in range(k)]
    votes = Counter(a for a in answers if a is not None)
    print("votes:", dict(votes), "| unfinished chains:", answers.count(None))
    return votes.most_common(1)[0][0] if votes else None

print("k=1:", self_consistent(cot_prompt, k=1))
print("k=7:", self_consistent(cot_prompt, k=7))

votes: {0.05: 1} | unfinished chains: 0
k=1: 0.05
votes: {0.05: 7} | unfinished chains: 0
k=7: 0.05


### Exercise 4: RAG
1. Add **two new facts** of your own to `docs` (re-fit the vectorizer after).
2. Ask a question that only your facts can answer.
3. Compare the answer with and without retrieval. What happens when the answer is **not** in the docs? Edit the RAG prompt so the model says *"I don't know"* in that case.

In [10]:
# 1) Add two facts of our own, then re-fit the vectorizer
docs += [
    "Acme Corp gives a 2-year warranty on all hardware products.",
    "Acme Corp's Basic plan is free and includes 1 seat.",
]
vec = TfidfVectorizer().fit(docs)
doc_vecs = vec.transform(docs)

# 2) A question that only our new facts can answer
q = "How long is the warranty on Acme Corp hardware?"
print("No context:", generate(f"Question: {q}\nAnswer:", max_tokens=60))
print("With RAG  :", rag_answer(q))

# 3) A question whose answer is NOT in the docs: the basic prompt may hallucinate
q_missing = "Who founded Acme Corp?"
print("\nNot in docs, basic RAG prompt:", rag_answer(q_missing))

# 4) Safer prompt: tell the model to admit when the context has no answer
def rag_answer_safe(query, k=1):
    context = "\n".join(retrieve(query, k))
    prompt = ("Answer using only the context below. If the context does not contain the answer, "
              "reply exactly: I don't know.\n\n"
              f"Context:\n{context}\n\nQuestion: {query}\nAnswer:")
    return generate(prompt)

print("Not in docs, safe RAG prompt :", rag_answer_safe(q_missing))
print("In docs, safe RAG prompt     :", rag_answer_safe(q))

No context: Please provide me with more context or a document. I do not have access to specific company warranty information like that for Acme Corp hardware.
With RAG  : Acme Corp gives a 2-year warranty on all hardware products.

Not in docs, basic RAG prompt: The context does not mention who founded Acme Corp.
Not in docs, safe RAG prompt : I don't know.
In docs, safe RAG prompt     : Acme Corp gives a 2-year warranty on all hardware products.


### Challenge (optional)
Pick a task from your own work (e.g. classify support tickets, summarize, translate to Khmer). Design a prompt, then improve it by adding **a role, a format constraint, and examples**. Note which change helped most.

---
## Recap
- **Few-shot**: examples fix format and style
- **CoT**: reasoning steps help multi-step problems
- **Self-consistency**: voting across samples improves reliability, at k× the cost
- **RAG**: grounds answers in your own data

Next: when prompting is not enough, **fine-tuning** updates the weights.

### Challenge: classify support tickets

Task from my own work: label a customer support ticket as `Billing`, `Technical` or `Account`. I improve one prompt in three steps (role, then format constraint, then examples) and compare the results.

In [11]:
tickets = [
    "I was charged twice for my Pro plan this month.",
    "The app crashes every time I open the reports page.",
    "I forgot my password and the reset email never arrives.",
    "Please send me an invoice for last quarter.",
    "Error 500 when I upload a file larger than 10 MB.",
    "How do I change the email address on my profile?",
]
expected = ["Billing", "Technical", "Account", "Billing", "Technical", "Account"]

v1 = lambda t: f"Classify this support ticket: {t}"                       # plain instruction
v2 = lambda t: ("You are a support triage assistant.\n"                    # + role + format constraint
                "Classify the ticket as exactly one of: Billing, Technical, Account.\n"
                f"Reply with the label only.\n\nTicket: {t}\nLabel:")
v3 = lambda t: ("You are a support triage assistant.\n"                    # + examples
                "Classify the ticket as exactly one of: Billing, Technical, Account.\n"
                "Reply with the label only.\n\n"
                "Ticket: My card was declined at renewal.\nLabel: Billing\n\n"
                "Ticket: The dashboard shows a blank screen.\nLabel: Technical\n\n"
                "Ticket: I want to delete my account.\nLabel: Account\n\n"
                f"Ticket: {t}\nLabel:")

for name, build in [("v1 plain", v1), ("v2 role+format", v2), ("v3 role+format+examples", v3)]:
    preds = [generate(build(t), max_tokens=8) for t in tickets]
    clean = [next((l for l in ("Billing", "Technical", "Account") if l.lower() in p.lower()), "?") for p in preds]
    acc = sum(c == e for c, e in zip(clean, expected)) / len(expected)
    print(f"{name:25s} accuracy={acc:.0%}  exact-format={sum(p.strip() in ('Billing','Technical','Account') for p in preds)}/{len(preds)}")

v1 plain                  accuracy=0%  exact-format=0/6
v2 role+format            accuracy=100%  exact-format=6/6
v3 role+format+examples   accuracy=100%  exact-format=6/6


**What helped most:** adding a **role and a format constraint** (v1 → v2).

| Prompt version | Accuracy | Exact one-word output |
|---|---|---|
| v1: plain instruction | 0% | 0/6 |
| v2: + role + format constraint | 100% | 6/6 |
| v3: + 3 examples | 100% | 6/6 |

- The plain prompt ("Classify this support ticket") made the model write sentences instead of a label, so none of the outputs were in the format I needed.
- Saying "You are a support triage assistant", listing the only allowed labels and adding "Reply with the label only" fixed both the format and the accuracy in one step.
- The examples (v3) did not change the score here, because my six tickets were clear-cut. I expect them to matter more for borderline tickets, for example a ticket that mentions both a payment and a login problem.
- Lesson: for classification, first constrain the output (role + allowed labels + "label only"), and only add examples if the model still gets edge cases wrong.